In [ ]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit > /dev/null

In [ ]:
import pandas as pd
from src import config, cedar

# Filter switches
REQUIRE_POSITIVE_ASSAY = True   # True: neoantigen needs >=1 "Positive*" T cell assay
                                # False: keep all melanoma neoantigen rows (more data, noisier positives)
SIM_ID = 0.4                    # CD-HIT identity (its minimum)
MAX_LEN = 35                    # None disables the length filter
SEED = 42
REFRESH = False                 # True: download a fresh CEDAR snapshot (CEDAR changes over time)

print("DATA_DIR:", config.DATA_DIR)
raw = cedar.get_raw(config.DATA_DIR, refresh=REFRESH)   # data_dir/cedar_raw_YYYYMMDD.csv
print(raw.groupby("label")[["has_positive_assay"]].agg(["size", "sum"]))

In [ ]:
df, info = cedar.build_initial(raw, require_positive_assay=REQUIRE_POSITIVE_ASSAY)
print(info)
print(f"{'class':<12}{'data':>8}")
for lab, name in [(1, "neoantigen"), (0, "other")]:
    print(f"{name:<12}{(df.label == lab).sum():>8}")
print(df["length"].describe())
assert df["sequence"].str.fullmatch(cedar.STD_AA).all()

In [ ]:
# 1) similarity filter (CD-HIT)  2) length filter  3) stratified 80/20 split
sim = cedar.cdhit_filter(df, config.WORK_DIR, sim_id=SIM_ID)
ln = (sim if MAX_LEN is None else sim[sim["length"] <= MAX_LEN]).reset_index(drop=True)
train, test = cedar.split(ln, test_size=0.2, seed=SEED)
final = pd.concat([train, test], ignore_index=True)

tables = cedar.stats_tables(df, sim, ln, train, test, SIM_ID, MAX_LEN)
for title, t in zip(["Table 1. The statistics of the survey dataset.", "", ""], tables):
    if title: print(title)
    print(t.to_string(index=False), "\n")

In [ ]:
# Save to Drive (data only; never git)
params = dict(REQUIRE_POSITIVE_ASSAY=REQUIRE_POSITIVE_ASSAY, SIM_ID=SIM_ID, MAX_LEN=MAX_LEN, SEED=SEED, **info)
cedar.save_outputs(config.DATA_DIR, df, final, tables, params)
!ls -la {config.DATA_DIR}